# Assignment 1

Computer Vision — COMP9320

In [ ]:
# Import the libraries used throughout the assignment.
%matplotlib inline
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Set the input and output paths relative to the project or notebook directory.
ASSIGNMENT_DIR = Path.cwd()
if not (ASSIGNMENT_DIR / "data" / "uni.jpeg").is_file():
    ASSIGNMENT_DIR = ASSIGNMENT_DIR / "Assignments" / "Assign_1"
DATA_DIR = ASSIGNMENT_DIR / "data"
OUTPUT_DIR = ASSIGNMENT_DIR / "outputs"

## Task 1: Image Loading and Grayscale Conversion

### 1. Use OpenCV to read the image uni.jpeg.

In [ ]:
# Read the original color image in OpenCV BGR format.
image_path = DATA_DIR / "uni.jpeg"
original_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
if original_bgr is None:
    raise FileNotFoundError(f"Unable to read image: {image_path}")

### 2. Display the original image and report its properties.

#### Spatial dimensions

- **Pixel:** one spatial sample of a digital image; a color pixel contains values for its color channels.
- **Spatial dimensions:** image height (rows) and width (columns), measured in pixels. For a color array, `shape` is `(height, width, channels)`; `shape[:2]` selects only the spatial dimensions.
- **BGR and RGB:** channel orders for blue, green, and red values. OpenCV loads this image in BGR order; Matplotlib expects RGB. Reordering channels preserves the spatial dimensions and displays the intended colors.

In [ ]:
# Display the original image in RGB order and report its height and width.
original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
height, width = original_bgr.shape[:2]
print(f"Spatial dimensions (height x width): {height} x {width} pixels")

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(original_rgb)
ax.set_title("Original image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
plt.tight_layout()
plt.show()

**Observation:** The image has 476 rows and 645 columns, so it is wider than it is tall. Array indexing uses `(row, column)`; the plot uses columns horizontally and rows vertically.

**References — documentation and further reading:**
- [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html) — pixel values, BGR order, and array shape.
- [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — RGB display and image coordinates.

#### Number of color channels

A **color channel** is a two-dimensional array containing one color component at every pixel. This image has blue, green, and red channels. OpenCV stores them in BGR order; each pixel therefore contains three values. The third element of `shape` gives the channel count, whereas `ndim` counts array axes.

In [ ]:
# Report the color channel count, channel order, and array structure.
channel_count = original_bgr.shape[2]
print(f"Number of color channels: {channel_count}")
print("OpenCV channel order: BGR (Blue, Green, Red)")
print("Display channel order: RGB (Red, Green, Blue)")
print(f"Image array shape (height, width, channels): {original_bgr.shape}")
print(f"Number of array axes: {original_bgr.ndim}")
for index, name in enumerate(("Blue", "Green", "Red")):
    print(f"Channel {index}: {name}; spatial shape: {original_bgr[:, :, index].shape}")

**Observation:** There are three color channels, each with 476 rows and 645 columns. The channel axis stores color components; it is not an additional spatial dimension.

**Reference — documentation:** [OpenCV: Basic Operations on Images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html), sections on image properties and splitting channels.

#### Data type and numerical intensity range

- **Data type (`dtype`):** the representation used to store each channel value. `uint8` is an unsigned 8-bit integer: it represents 256 integer levels from 0 to 255.
- **Channel intensity:** the stored value of one color component at a pixel. Zero is the minimum component level; 255 is the maximum. A single channel value is not the overall brightness of a color pixel.
- **Representable range:** the limits allowed by the data type, obtained with `np.iinfo`. **Observed range:** the minimum and maximum actually present in the image, obtained with `min()` and `max()`. These ranges need not be equal.

In [ ]:
# Report the data type, representable limits, and observed channel intensities.
dtype_limits = np.iinfo(original_bgr.dtype)
print(f"Data type: {original_bgr.dtype}")
print(f"Bits per channel value: {dtype_limits.bits}")
print(f"Representable intensity range: [{dtype_limits.min}, {dtype_limits.max}]")
print(f"Representable intensity levels: {int(dtype_limits.max) - int(dtype_limits.min) + 1}")
print(f"Observed intensity range across all channels: [{original_bgr.min()}, {original_bgr.max()}]")
for index, name in enumerate(("Blue", "Green", "Red")):
    channel = original_bgr[:, :, index]
    print(f"Observed {name} channel range: [{channel.min()}, {channel.max()}]")

**Observation:** The image uses `uint8`. The observed range is [0, 255] both overall and in each color channel. Each channel contains at least one minimum and one maximum value; this does not establish that all 256 levels occur or that a completely black or white pixel exists.

**Reference — documentation:** [NumPy: numpy.iinfo](https://numpy.org/doc/stable/reference/generated/numpy.iinfo.html) — integer limits and bit depth.

#### Memory requirements

- **Byte:** 8 bits. A `uint8` channel value occupies one byte (`itemsize`).
- **Array data memory:** the bytes occupied by the decoded pixel values (`nbytes`), excluding Python object metadata and other arrays. For this image: $M = H \times W \times C \times \text{bytes per channel value}$.
- **Units:** 1 KiB = 1,024 bytes; 1 MiB = 1,048,576 bytes; 1 MB = 1,000,000 bytes.
- **JPEG file size:** compressed storage on disk; it differs from the decoded array memory.

In [ ]:
# Calculate decoded image memory and compare it with the compressed JPEG file size.
pixel_count = height * width
bytes_per_value = original_bgr.itemsize
expected_bytes = pixel_count * channel_count * bytes_per_value
image_bytes = original_bgr.nbytes
jpeg_bytes = image_path.stat().st_size
print(f"Number of spatial pixels: {pixel_count:,}")
print(f"Number of stored channel values: {original_bgr.size:,}")
print(f"Bytes per channel value: {bytes_per_value}")
print(f"Bytes per color pixel: {channel_count * bytes_per_value}")
print(f"Memory calculation: {height} x {width} x {channel_count} x {bytes_per_value} = {expected_bytes:,} bytes")
print(f"Decoded array data (nbytes): {image_bytes:,} bytes")
print(f"Decoded array data: {image_bytes / 1024:.2f} KiB; {image_bytes / 1024**2:.4f} MiB; {image_bytes / 10**6:.5f} MB")
print(f"Compressed JPEG file: {jpeg_bytes:,} bytes ({jpeg_bytes / 1024:.2f} KiB)")
print(f"Calculated memory matches nbytes: {expected_bytes == image_bytes}")

**Observation:** The decoded BGR pixel data occupies 921,060 bytes, whereas the JPEG file occupies 67,589 bytes. JPEG stores a compressed representation. The reported array memory excludes the separate RGB display copy, plotting buffers, and Python object overhead.

**References — documentation:** [NumPy: ndarray.nbytes](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.nbytes.html) and [ndarray.itemsize](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.itemsize.html).

### 3. Convert the original image to grayscale.

**Grayscale** represents each pixel with one intensity value instead of three color components. For `uint8`, 0 represents black and 255 white. OpenCV combines the channels using:

$$Y = 0.299R + 0.587G + 0.114B.$$

The unequal weights approximate the different contributions of the color components to perceived brightness; this is not a simple channel average or a physical luminance measurement. `COLOR_BGR2GRAY` matches the input BGR order. Integer conversion quantizes the result, so manually computed floating-point values may differ slightly.

The conversion preserves height and width but discards color information; the original colors cannot generally be recovered from the grayscale image.

In [ ]:
# Convert the original BGR image to one grayscale channel and report its properties.
gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
print("Conversion: cv2.COLOR_BGR2GRAY")
print("Weighted intensity: Y = 0.299R + 0.587G + 0.114B")
print(f"Grayscale shape (height, width): {gray.shape}")
print("Number of grayscale channels: 1")
print(f"Number of array axes: {gray.ndim}")
print(f"Data type: {gray.dtype}")
print(f"Observed intensity range: [{gray.min()}, {gray.max()}]")
print(f"Spatial dimensions preserved: {gray.shape == original_bgr.shape[:2]}")
print(f"Grayscale array data: {gray.nbytes:,} bytes")

**Observation:** The grayscale image is a two-dimensional `uint8` array of shape (476, 645), with intensities from 0 to 255. It stores one value per pixel and uses 307,020 bytes of pixel data, one-third of the original BGR array.

**Reference — documentation:** [OpenCV: Color conversions, RGB ↔ GRAY](https://docs.opencv.org/4.x/de/d25/imgproc_color_conversions.html) — conversion formula and channel-order flags.

### 4. Display the grayscale image.

A **colormap** maps stored values to display colors. `cmap="gray"` displays low intensities as dark and high intensities as light. Setting `vmin=0` and `vmax=255` fixes the display scale to the full 8-bit range; it does not modify the image values. `interpolation="nearest"` avoids smoothing between pixels during rendering.

In [ ]:
# Display the grayscale image with a fixed intensity scale and labeled axes.
fig, ax = plt.subplots(figsize=(8, 6))
gray_plot = ax.imshow(gray, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
ax.set_title("Grayscale image — uni.jpeg")
ax.set_xlabel("Column (pixels)")
ax.set_ylabel("Row (pixels)")
fig.colorbar(gray_plot, ax=ax, label="Grayscale intensity (0–255)")
plt.tight_layout()
plt.show()

**Observation:** The grayscale view retains the spatial layout while removing color distinctions. Dark and light areas represent low and high weighted intensities. Different colors can map to similar gray values, so some color-based distinctions may be lost.

**Reference — documentation:** [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — colormaps, intensity limits, and interpolation.

## Task 2: Edge Detection Using Canny

### 1. Use the grayscale image obtained in Task 1.

An **edge** is a location of a rapid spatial change in image intensity. It may indicate an object boundary, texture, or a lighting change; not every detected edge is an object boundary. **Edge detection** identifies these changes to emphasize image structure.

Reuse `gray` from Task 1 as the single-channel input. No image reload or second grayscale conversion is needed.

In [ ]:
# Reuse the Task 1 grayscale image and validate the input for edge detection.
assert gray.ndim == 2, "Expected a single-channel grayscale image."
assert gray.dtype == np.uint8, "Expected an 8-bit unsigned integer image."
assert gray.shape == original_bgr.shape[:2], "Spatial dimensions do not match."
canny_input = gray
print("Input source: Task 1 grayscale image (gray)")
print(f"Input shape (height, width): {canny_input.shape}")
print("Number of channels: 1")
print(f"Input data type: {canny_input.dtype}")
print(f"Observed intensity range: [{canny_input.min()}, {canny_input.max()}]")
print(f"Same array as Task 1 grayscale image: {canny_input is gray}")

**Observation:** The input is the existing (476, 645) `uint8` grayscale array. `canny_input = gray` creates another reference to the same array, not a copy; this step does not change its values.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — intensity changes and the Canny pipeline.

### 2. Apply the Canny edge detection algorithm to extract the edges.
,
,**Gaussian smoothing** averages nearby pixels with greater weight near the center to reduce noise before detecting intensity changes. We explicitly apply a 5 × 5 Gaussian kernel with standard deviation $\sigma=1$ pixel.
,
,A **gradient** measures the spatial change in intensity. Canny uses gradients, thins candidate edges, then links them using **hysteresis**: strong responses above the upper threshold seed edges; weaker candidates between the thresholds survive only when connected to strong edges. Thresholds apply to gradient magnitude, not directly to grayscale values.
,
,Use initial thresholds of 50 and 150. `apertureSize=3` selects a 3 × 3 Sobel derivative kernel; `L2gradient=True` uses $\sqrt{G_x^2+G_y^2}$. The threshold choice will be assessed in Steps 5–6. A **binary edge map** contains two labels: 255 for detected edges and 0 for non-edges.

**Parameter glossary:**

| Term / variable | Meaning in this implementation |
| --- | --- |
| Kernel | A small matrix of filter weights applied over a pixel neighborhood. |
| Sobel operator | A discrete derivative filter approximating intensity changes along columns ($G_x$) and rows ($G_y$). It combines differentiation with smoothing in the perpendicular direction. |
| `sobel_aperture` | Our variable for the Sobel kernel side length; passed to Canny as `apertureSize`. A value of 3 means a 3 × 3 neighborhood, not three passes or a camera aperture. |
| `gaussian_kernel` | Width and height of the separate Gaussian smoothing kernel: (5, 5). |
| `gaussian_sigma` | Gaussian standard deviation in pixels; controls the spread of smoothing weights. |
| `use_l2_gradient` | Our Boolean variable passed as `L2gradient`; `True` selects Euclidean gradient magnitude. |
| `BORDER_REFLECT_101` | Extends image values by reflection without repeating the boundary pixel, allowing filtering near image borders. |

**Reference — documentation:** [OpenCV: Sobel Derivatives](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — derivative kernels and gradient estimation. The Sobel kernel and the separate Gaussian kernel have different roles.

In [ ]:
# Smooth the grayscale input, apply Canny, and report the settings and output.
gaussian_kernel = (5, 5)
gaussian_sigma = 1.0
lower_threshold = 50
upper_threshold = 150
sobel_aperture = 3
use_l2_gradient = True
smoothed_gray = cv2.GaussianBlur(
    canny_input, gaussian_kernel, sigmaX=gaussian_sigma,
    sigmaY=gaussian_sigma, borderType=cv2.BORDER_REFLECT_101
)
edges = cv2.Canny(
    smoothed_gray, lower_threshold, upper_threshold,
    apertureSize=sobel_aperture, L2gradient=use_l2_gradient
)
edge_pixel_count = np.count_nonzero(edges)
print(f"Gaussian kernel: {gaussian_kernel}; sigma: {gaussian_sigma} pixel")
print("Gaussian border handling: BORDER_REFLECT_101")
print(f"Hysteresis thresholds (lower, upper): ({lower_threshold}, {upper_threshold})")
print(f"Sobel aperture size: {sobel_aperture}; L2gradient: {use_l2_gradient}")
print(f"Edge map shape: {edges.shape}; data type: {edges.dtype}")
print(f"Unique output values: {np.unique(edges).tolist()}")
print(f"Detected edge pixels: {edge_pixel_count:,} / {edges.size:,}")
print(f"Edge pixel percentage: {100 * edge_pixel_count / edges.size:.2f}%")

**Interpretation:** White pixels label retained edges; black pixels label non-edges. The edge pixel percentage describes output density, not detection accuracy. The same smoothing and gradient settings must be retained when comparing thresholds.

**References — documentation and further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) and [Canny API](https://docs.opencv.org/4.x/dd/d1a/group__imgproc__feature.html) — algorithm and parameters.

### 3. Display the grayscale image and the resulting edge image side by side.

The **grayscale image** stores intensity values, whereas the **edge map** stores binary edge labels. Both panels use the same spatial coordinates and a fixed 0–255 display range. The left panel shows the original Task 1 grayscale image; Canny was applied to its Gaussian-smoothed version in Step 2.

In [ ]:
# Compare the original grayscale image with the Canny edge map at matching scales.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
axes[0].imshow(canny_input, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Original grayscale image")
axes[1].imshow(edges, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[1].set_title(f"Canny edges — thresholds {lower_threshold}, {upper_threshold}")
for ax in axes:
    ax.set_xlabel("Column (pixels)")
    ax.set_ylabel("Row (pixels)")
fig.suptitle("Grayscale and Canny edge comparison")
plt.tight_layout()
plt.show()

**Observation:** The edge map highlights building outlines, paths, and fine vegetation or architectural texture while discarding the grayscale shading. Some visible responses arise from texture and shadows rather than object boundaries. White pixels represent detected edges, not brighter scene regions; visual comparison alone does not establish detection accuracy.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — grayscale and edge-map comparison.

### 4. Explain the five fundamental stages of the Canny algorithm.

#### 4.1 Gaussian smoothing

**Smoothing** reduces rapid local intensity variations. A Gaussian filter replaces each pixel with a weighted average of its neighborhood, assigning larger weights near the center:

$$K(i,j)=\frac{\exp\left[-(i^2+j^2)/(2\sigma^2)\right]}{\sum_{u,v}\exp\left[-(u^2+v^2)/(2\sigma^2)\right]},\qquad I_s(x,y)=\sum_{i,j}K(i,j)I(x-i,y-j).$$

Here, $I$ is the grayscale image, $I_s$ is the smoothed image, and $(i,j)$ are kernel offsets. **Normalization** makes the finite kernel weights sum to 1, preserving constant intensities away from numerical rounding.

**Why before differentiation?** Derivatives respond strongly to rapid variations, including noise. Smoothing suppresses some unwanted responses, but also softens real edges and fine texture. Larger $\sigma$ generally increases this tradeoff when the kernel support is sufficient.

**Connection to Step 2:** `cv2.GaussianBlur` explicitly produces `smoothed_gray` with a 5 × 5 kernel and $\sigma_x=\sigma_y=1$. The separate Gaussian operation is not an additional Gaussian stage automatically applied by `cv2.Canny`. Sobel's built-in perpendicular smoothing does not replace this explicit preprocessing.

**References — documentation:** [OpenCV: Smoothing Images](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html) — Gaussian kernels and sigma; [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — smoothing before gradient computation.

#### 4.2 Gradient computation

A **partial derivative** measures change along one coordinate while holding the other fixed. The image **gradient** is the vector $\nabla I_s=(G_x,G_y)$: $G_x$ approximates change along columns (left to right), and $G_y$ along rows (top to bottom). Sobel estimates these derivatives from neighboring pixels of the smoothed image.

For the 3 × 3 Sobel operator, the correlation kernels are:

$$K_x=\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix},\qquad K_y=\begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix}.$$

**Correlation** multiplies each kernel weight by the corresponding neighborhood value and sums the products without flipping the kernel. Thus $G_x$ is positive for intensity increasing to the right, while $G_y$ is positive for intensity increasing downward. A large absolute response indicates a strong change; $G_x$ emphasizes vertical boundaries and $G_y$ horizontal boundaries.

**Code parameters:** `dx` and `dy` are derivative orders: `(1, 0)` computes the first x derivative, and `(0, 1)` the first y derivative. `ksize=sobel_aperture` selects the 3 × 3 kernel. `ddepth=cv2.CV_32F` stores signed 32-bit floating-point responses, preserving negative derivatives that `uint8` cannot represent. `BORDER_REPLICATE` extends the nearest boundary value for derivative filtering.

The following cell illustrates this stage separately; it does not replace the existing `cv2.Canny` result or claim to expose its internal buffers.

In [ ]:
# Compute signed Sobel derivatives of the Gaussian-smoothed image for illustration.
gradient_x = cv2.Sobel(
    smoothed_gray, cv2.CV_32F, dx=1, dy=0,
    ksize=sobel_aperture, borderType=cv2.BORDER_REPLICATE
)
gradient_y = cv2.Sobel(
    smoothed_gray, cv2.CV_32F, dx=0, dy=1,
    ksize=sobel_aperture, borderType=cv2.BORDER_REPLICATE
)
print(f"Sobel kernel size: {sobel_aperture} x {sobel_aperture}")
print("Derivative border handling: BORDER_REPLICATE")
for name, derivative in (("Gx: column-direction change", gradient_x),
                         ("Gy: row-direction change", gradient_y)):
    print(f"{name}; shape: {derivative.shape}; dtype: {derivative.dtype}")
    print(f"  Signed response range: [{derivative.min():.1f}, {derivative.max():.1f}]")

**Interpretation:** Derivative responses are signed changes, not grayscale intensities; their range is not restricted to 0–255. Negative responses indicate decreasing intensity in the corresponding coordinate direction. The next stage combines these components into magnitude and orientation.

**Reference — documentation:** [OpenCV: Sobel Derivatives](https://docs.opencv.org/4.x/d2/d2c/tutorial_sobel_derivatives.html) — derivative kernels, gradient components, and output depth.

#### 4.3 Gradient magnitude and orientation estimation

**Magnitude** is the length of the gradient vector and measures the strength of local intensity change. **Orientation** is the direction of greatest increase in intensity, perpendicular to the local edge tangent:

$$M=\sqrt{G_x^2+G_y^2},\qquad \theta=\operatorname{atan2}(G_y,G_x).$$

`np.hypot(Gx, Gy)` computes the Euclidean magnitude element by element. `np.arctan2(Gy, Gx)` uses both signs to select the correct quadrant and avoids explicit division by zero. Its argument order is y first, x second. It returns **radians** (a full turn is $2\pi$); `np.degrees` converts to **degrees** (a full turn is 360°).

Since image rows increase downward, 0° points right, +90° down, and −90° up. Where magnitude is zero, orientation is undefined; the code marks it as `NaN` (not a number). **Modulo 180°** maps opposite gradient directions to the same unoriented axis, useful for comparing neighbors during non-maximum suppression. These illustrative arrays do not expose Canny's internal buffers.

In [ ]:
# Calculate gradient strength and direction while excluding zero-gradient orientations.
gradient_magnitude = np.hypot(gradient_x, gradient_y)
orientation_degrees = np.degrees(np.arctan2(gradient_y, gradient_x))
valid_orientation = gradient_magnitude > 0
orientation_degrees = np.where(valid_orientation, orientation_degrees, np.nan)
orientation_axis = np.mod(orientation_degrees, 180.0)
print(f"Magnitude shape: {gradient_magnitude.shape}; dtype: {gradient_magnitude.dtype}")
print(f"Magnitude range: [{gradient_magnitude.min():.2f}, {gradient_magnitude.max():.2f}]")
print("Directed angle convention: degrees from +x; positive toward increasing rows")
print("Directed angle interval: [-180, 180] degrees")
print("Unoriented gradient axis interval: [0, 180) degrees")
print(f"Pixels with defined orientation: {np.count_nonzero(valid_orientation):,}")
print(f"Zero-gradient pixels (undefined orientation): {np.count_nonzero(~valid_orientation):,}")
if np.any(valid_orientation):
    print(f"Observed directed angle range: [{np.nanmin(orientation_degrees):.2f}, {np.nanmax(orientation_degrees):.2f}] degrees")

**Interpretation:** Magnitude is nonnegative and can exceed 255 because it is derived from signed Sobel responses, not stored grayscale intensity. A large magnitude alone does not establish an edge: non-maximum suppression and hysteresis still determine which responses survive. At very small magnitudes, orientation is sensitive to small intensity changes.

**References — documentation:** [NumPy: hypot](https://numpy.org/doc/stable/reference/generated/numpy.hypot.html), [NumPy: arctan2](https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html), and [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html).

#### 4.4 Non-maximum suppression (NMS)

**Suppression** means setting a response to zero. A **local maximum** is a response at least as large as its nearby comparison values. NMS thins broad gradient ridges by retaining local peaks **along the gradient direction**, across the edge rather than along its tangent.

At pixel $p$, compare $M(p)$ with magnitudes on both sides along the gradient axis. Suppress it if either comparison is larger. A common discrete approximation groups orientations into 0°, 45°, 90°, and 135°:

| Gradient axis (image coordinates) | Neighbors compared |
| --- | --- |
| 0° | Left and right |
| 45° | Upper-left and lower-right |
| 90° | Above and below |
| 135° | Upper-right and lower-left |

**Tie handling** determines which equal neighboring peaks survive; implementations may use asymmetric comparisons. NMS retains selected magnitudes and zeros others; its conceptual output is not yet the final binary edge map. Hysteresis subsequently selects and links candidates.

The following one-dimensional profile illustrates a cross-section along the gradient direction. It uses strict comparisons for a unique peak and sets its two endpoints to zero; it is not a full-image NMS implementation or Canny's internal output.

In [ ]:
# Illustrate thinning of a gradient ridge using a unique local maximum.
nms_profile = np.array([0, 20, 60, 100, 70, 30, 0], dtype=np.float32)
nms_profile_thinned = np.zeros_like(nms_profile)
local_peak_mask = (
    (nms_profile[1:-1] > nms_profile[:-2])
    & (nms_profile[1:-1] > nms_profile[2:])
)
nms_profile_thinned[1:-1] = np.where(local_peak_mask, nms_profile[1:-1], 0)
print("Illustrative gradient magnitudes:", nms_profile.tolist())
print("After non-maximum suppression:", nms_profile_thinned.tolist())
print(f"Nonzero responses before: {np.count_nonzero(nms_profile)}")
print(f"Nonzero responses after: {np.count_nonzero(nms_profile_thinned)}")

**Interpretation:** The response 100 survives because it exceeds both neighbors. The other positive responses are suppressed, reducing a five-sample ridge to one sample. No lower or upper threshold is applied in this example; NMS compares neighboring strengths rather than selecting edges by absolute strength.

**Reference — documentation:** [OpenCV: Canny Edge Detection, Non-maximum Suppression](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — comparisons along the gradient direction.

#### 4.5 Hysteresis thresholding

**Hysteresis** uses two thresholds and connectivity to decide which thinned gradient responses form final edges. With lower threshold $T_L$ and upper threshold $T_H$ ($T_L<T_H$), this example uses the following explicit convention:

- **Strong edge seed:** a retained magnitude greater than $T_H$.
- **Weak candidate:** a retained magnitude greater than $T_L$ but no greater than $T_H$.
- **Rejected response:** a magnitude no greater than $T_L$.

**8-connectivity** allows a pixel to connect to its horizontal, vertical, and diagonal neighbors. Keep a weak candidate only if a path of eligible candidates connects it to a strong seed. The path may contain several weak pixels: direct adjacency to a strong seed is not required. An isolated weak component is discarded.

The thresholds apply to magnitudes after non-maximum suppression, not to the original grayscale intensities. The final **binary edge map** uses 255 for retained pixels and 0 otherwise.

The illustrative matrix below represents responses *after* NMS. `connectedComponents` assigns a distinct integer **label** to each connected foreground region; label 0 is background. Keeping components containing strong seeds illustrates hysteresis without reproducing Canny's internal implementation.

In [ ]:
# Illustrate hysteresis by keeping candidate components connected to strong seeds.
hysteresis_demo = np.array([
    [0, 200, 90, 70, 0, 0, 0],
    [0,   0,  0,  0, 0, 0, 0],
    [0,   0,  0,  0, 0, 0, 0],
    [0,  80, 60,  0, 0, 0, 0],
    [0,   0,  0,  0, 0, 0, 0]
], dtype=np.float32)
demo_low, demo_high = 50, 150
candidate_mask = hysteresis_demo > demo_low
strong_mask = hysteresis_demo > demo_high
component_count, component_labels = cv2.connectedComponents(
    candidate_mask.astype(np.uint8), connectivity=8
)
strong_component_labels = np.unique(component_labels[strong_mask])
keep_mask = candidate_mask & np.isin(component_labels, strong_component_labels)
hysteresis_result = (keep_mask.astype(np.uint8) * 255)
print(f"Illustrative thresholds: lower={demo_low}, upper={demo_high}")
print("Post-NMS magnitude example:\n", hysteresis_demo)
print("Final binary example:\n", hysteresis_result)
print(f"Candidate components (excluding background): {component_count - 1}")
print(f"Strong seeds: {np.count_nonzero(strong_mask)}")
print(f"Weak candidates retained: {np.count_nonzero(keep_mask & ~strong_mask)}")
print(f"Weak candidates discarded: {np.count_nonzero(candidate_mask & ~strong_mask & ~keep_mask)}")

**Interpretation:** The seed 200 retains the connected chain 90 → 70, even though 70 does not directly touch the seed. The disconnected weak responses 80 and 60 are discarded. Thus hysteresis can preserve weaker continuations while rejecting isolated weak responses. Connected noise or texture can still survive; connectivity does not guarantee an object boundary.

**Connection to Step 2:** `lower_threshold=50` and `upper_threshold=150` supply the two thresholds to `cv2.Canny`. The five stages are now explained; the following steps assess the selected thresholds and their effects.

**Reference — documentation:** [OpenCV: Canny Edge Detection, Hysteresis Thresholding](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — strong seeds and connected weak candidates.

### 5. Report suitable lower and upper hysteresis thresholds and justify their selection.

**Selected thresholds:** $T_L=50$ and $T_H=150$, with the existing 5 × 5 Gaussian smoothing ($\sigma=1$), 3 × 3 Sobel aperture, and L2 magnitude.

**Selection criterion:** retain recognizable building and path contours while limiting dense fine-texture responses. Visual inspection of the selected edge map shows a useful compromise: major outlines remain visible, although vegetation, windows, and shadows still produce responses. A lower pair (25, 75) retains noticeably denser texture; a higher pair (100, 200) suppresses more detail and weak contour segments. These are qualitative comparisons, not measured error rates.

The upper threshold selects strong seeds; the lower threshold admits weaker connected continuations. Their 3:1 ratio is a practical choice for this image and preprocessing, not a universal rule. Thresholds are on the Sobel gradient scale, so 150 is not a grayscale brightness cutoff. Changing smoothing, derivative aperture, or magnitude definition requires reassessing them.

No annotated reference edge map is available; therefore, the choice is visually suitable rather than proven optimal. Edge density alone cannot establish accuracy.

In [ ]:
# Report the selected thresholds and the fixed preprocessing used to justify them.
print(f"Selected lower threshold: {lower_threshold}")
print(f"Selected upper threshold: {upper_threshold}")
print(f"Upper-to-lower threshold ratio: {upper_threshold / lower_threshold:.1f}:1")
print(f"Gaussian kernel: {gaussian_kernel}; sigma: {gaussian_sigma}")
print(f"Sobel aperture: {sobel_aperture}; L2gradient: {use_l2_gradient}")
print(f"Selected edge pixels: {np.count_nonzero(edges):,} / {edges.size:,}")
print(f"Selected edge density: {100 * np.count_nonzero(edges) / edges.size:.2f}%")
print("Selection basis: visual balance between contour retention and texture suppression.")
print("Validation limit: no ground-truth edge labels; optimality is not established.")

**Reference — documentation:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — lower/upper thresholds and connectivity. The numerical selection and visual assessment above are specific to `uni.jpeg`, not prescribed by the documentation.

### 6. Investigate the effect of changing the lower and upper thresholds.

#### 6.1 Edge connectivity

**Connectivity** describes whether edge pixels form continuous paths. A **connected component** is a group linked by 8-neighbor paths. We report component count and largest component size (in pixels) as descriptive summaries, not accuracy measures: unrelated edges may merge, and legitimate objects may form separate components.

**Controlled comparison:** use the same `smoothed_gray`, Sobel aperture, and L2 setting. In the first row, vary only the lower threshold while fixing the upper at 150; in the second row, vary only the upper while fixing the lower at 50. The baseline (50, 150) appears in both rows. A **threshold sweep** means repeating the detector over specified threshold values.

In [ ]:
# Sweep one threshold at a time and summarize 8-connected edge components.
threshold_sweeps = [
    [(25, 150), (50, 150), (100, 150)],
    [(50, 75), (50, 150), (50, 225)]
]
connectivity_results = {}
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
print(f"{'Lower':>6} {'Upper':>6} {'Edge pixels':>12} {'Density %':>10} {'Components':>12} {'Largest (px)':>13}")
for row, pairs in enumerate(threshold_sweeps):
    for column, (low, high) in enumerate(pairs):
        trial_edges = cv2.Canny(smoothed_gray, low, high,
                               apertureSize=sobel_aperture, L2gradient=use_l2_gradient)
        count, labels, stats, centroids = cv2.connectedComponentsWithStats(
            trial_edges, connectivity=8
        )
        component_sizes = stats[1:, cv2.CC_STAT_AREA]
        largest_component = int(component_sizes.max()) if component_sizes.size else 0
        trial_count = int(np.count_nonzero(trial_edges))
        connectivity_results[(low, high)] = trial_edges
        print(f"{low:6d} {high:6d} {trial_count:12,d} {100 * trial_count / trial_edges.size:10.2f} {count - 1:12,d} {largest_component:13,d}")
        ax = axes[row, column]
        ax.imshow(trial_edges, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        ax.set_title(f"Lower={low}, upper={high}")
        ax.set_xlabel("Column (pixels)")
        ax.set_ylabel("Row (pixels)")
fig.suptitle("Threshold sweeps: top — upper fixed at 150; bottom — lower fixed at 50")
plt.tight_layout()
plt.show()

**Reading the output:** `connectedComponentsWithStats` returns labels and component statistics. Label 0 is background, so `count - 1` excludes it; `stats[1:, cv2.CC_STAT_AREA]` selects foreground component sizes. Size counts pixels, not geometric contour length. `centroids` are component mean positions and are unused here.

**Expected mechanism:** Lowering the lower threshold admits weaker links to existing strong seeds and can join fragments; raising it can remove links and fragment or delete contours. Lowering the upper threshold introduces more strong seeds and can retain previously unseeded candidate components; raising it can remove components lacking a sufficiently strong seed. These changes can increase or decrease the total component count, so that count alone cannot prove improved continuity.

**References — documentation:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — hysteresis; [OpenCV: Structural Analysis and Shape Descriptors](https://docs.opencv.org/4.x/d3/dc0/group__imgproc__shape.html) — `connectedComponentsWithStats` and `CC_STAT_AREA`.

**Observed results:** With upper=150, raising the lower threshold from 25 to 100 reduces edge pixels from 53,098 to 33,531, raises component count from 594 to 969, and reduces the largest component from 2,319 to 834 pixels, consistent with loss of weak connecting segments. With lower=50, raising the upper threshold from 75 to 225 reduces component count from 1,832 to 321 while the largest remains 1,547 pixels: many components lose their seeds, rather than the remaining contours necessarily becoming better connected.

#### 6.2 Edge localization

**Localization** is how closely a detected edge position matches the true intensity boundary. Here, smoothing and derivative settings remain fixed: changing hysteresis thresholds selects among the same thinned candidate positions rather than recomputing their locations. It can remove a weak contour or retain a nearby competing response, affecting which boundary is visible.

A **pixel mask** is a Boolean array marking selected coordinates. Compare a relaxed threshold pair with a stricter pair: count shared coordinates and strict-only coordinates. A zero strict-only count supports subset selection for these runs; it does not measure localization error against a true boundary.

For a magnified view, a **region of interest (ROI)** selects rows 160–299 and columns 250–429 using end-exclusive slices. Overlay colors distinguish shared edge pixels (white), pixels retained only at lower=25 (cyan), and pixels retained only at lower=100 (magenta).

In [ ]:
# Compare edge coordinates under stricter thresholds and magnify a fixed region.
ordered_threshold_pairs = [
    ((25, 150), (50, 150)), ((50, 150), (100, 150)),
    ((50, 75), (50, 150)), ((50, 150), (50, 225))
]
print(f"{'Relaxed -> stricter':>30} {'Shared pixels':>14} {'Stricter-only':>14}")
for relaxed_pair, strict_pair in ordered_threshold_pairs:
    relaxed_mask = connectivity_results[relaxed_pair] > 0
    strict_mask = connectivity_results[strict_pair] > 0
    shared_count = np.count_nonzero(relaxed_mask & strict_mask)
    strict_only_count = np.count_nonzero(strict_mask & ~relaxed_mask)
    print(f"{str(relaxed_pair) + ' -> ' + str(strict_pair):>30} {shared_count:14,d} {strict_only_count:14,d}")
roi_rows, roi_columns = slice(160, 300), slice(250, 430)
relaxed_roi = connectivity_results[(25, 150)][roi_rows, roi_columns] > 0
strict_roi = connectivity_results[(100, 150)][roi_rows, roi_columns] > 0
coordinate_overlay = np.zeros((*relaxed_roi.shape, 3), dtype=np.uint8)
coordinate_overlay[relaxed_roi & strict_roi] = [255, 255, 255]
coordinate_overlay[relaxed_roi & ~strict_roi] = [0, 255, 255]
coordinate_overlay[strict_roi & ~relaxed_roi] = [255, 0, 255]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(gray[roi_rows, roi_columns], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Grayscale ROI")
axes[1].imshow(coordinate_overlay, interpolation="nearest")
axes[1].set_title("White: shared; cyan: relaxed-only; magenta: strict-only")
for ax in axes:
    ax.set_xlabel("Column within ROI (pixels)")
    ax.set_ylabel("Row within ROI (pixels)")
plt.tight_layout()
plt.show()

**Interpretation:** In these controlled runs, stricter settings retain a subset of the relaxed edge coordinates (see the stricter-only counts). Shared responses remain at identical pixel positions, while weaker segments disappear. This is selection and loss of coverage, not evidence that the retained edges move toward a true boundary. Smoothing scale, derivative estimation, and non-maximum suppression govern the candidate positions. Without annotated boundaries, we cannot report localization error or claim that higher thresholds improve positional accuracy.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — directional suppression followed by hysteresis selection.

#### 6.3 False-positive detections

A **false positive (FP)** is a detected edge that does not match a target edge in a labeled reference. A **false negative (FN)** is a target edge that the detector misses. **Ground truth** is the annotated reference defining which boundaries count as targets.

Texture (repeated fine intensity changes) and shadows can produce genuine image gradients without indicating the object boundaries of interest. They are not automatically false positives: the judgment depends on the target definition. Here, visual inspection focuses on recognizable building and path boundaries; no pixel-level ground truth is available.

Lower thresholds can admit noise-driven responses and additional texture; higher thresholds suppress many weak responses but can also discard valid low-contrast boundaries. The following fixed ROI samples vegetation and foreground texture. It varies only the upper threshold (75, 150, 225), keeping the lower at 50 and all preprocessing unchanged. Counts of added or removed pixels describe sensitivity, not verified FP/FN counts.

In [ ]:
# Inspect textured foreground responses without treating extra edge pixels as confirmed errors.
texture_rows, texture_columns = slice(330, 460), slice(50, 300)
fp_pairs = [(50, 75), (50, 150), (50, 225)]
baseline_mask = connectivity_results[(50, 150)] > 0
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
axes[0].imshow(gray[texture_rows, texture_columns], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Grayscale texture ROI")
print("Texture ROI: rows 330–459, columns 50–299")
print(f"{'Lower':>6} {'Upper':>6} {'ROI edges':>10} {'ROI density %':>14} {'Added vs baseline':>18} {'Removed vs baseline':>20}")
for ax, pair in zip(axes[1:], fp_pairs):
    trial_mask = connectivity_results[pair] > 0
    roi_mask = trial_mask[texture_rows, texture_columns]
    added_pixels = np.count_nonzero(trial_mask & ~baseline_mask)
    removed_pixels = np.count_nonzero(baseline_mask & ~trial_mask)
    print(f"{pair[0]:6d} {pair[1]:6d} {np.count_nonzero(roi_mask):10,d} {100 * np.mean(roi_mask):14.2f} {added_pixels:18,d} {removed_pixels:20,d}")
    ax.imshow(roi_mask, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Edges: lower={pair[0]}, upper={pair[1]}")
for ax in axes:
    ax.set_xlabel("Column within ROI (pixels)")
    ax.set_ylabel("Row within ROI (pixels)")
plt.tight_layout()
plt.show()
print("Added/removed counts use the full image; ROI counts use the displayed crop.")
print("No annotated reference: FP counts, FN counts, and false-positive rates are not measured.")

**Interpretation:** Compare the cropped edge maps against the grayscale texture: stricter settings remove many responses in vegetation, but a cleaner-looking result alone does not establish fewer errors. The full-image threshold sweep also shows loss of weaker contours. Thus threshold selection balances additional responses against missing useful structure; the selected (50, 150) pair is a visual compromise for this image.

**Task 2 finding:** Thresholds change edge survival and connectivity. With fixed smoothing and derivatives, shared edge coordinates remain unchanged. Without labeled target boundaries, localization error and false-positive rates cannot be quantified from edge density or component counts.

**Reference — further reading:** [OpenCV: Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html) — noise sensitivity and threshold-dependent retention. The ROI assessment is a qualitative analysis of this image.

## Task 3: Gaussian Filtering and Comparison

### 1. Apply a Gaussian filter with standard deviation σ = 2√2.

A **Gaussian filter** smooths the image using a normalized, bell-shaped weighting kernel. **Standard deviation** $\sigma$ sets its spatial spread in pixels; **variance** is $\sigma^2$. Here $\sigma=2\sqrt{2}\approx2.8284$ pixels and variance is 8 pixels².

Use the original Task 1 grayscale image, not the Canny-smoothed input. For a reproducible finite kernel, choose radius $r=\lceil3\sigma\rceil$ and odd side length $k=2r+1$, giving 19 × 19. The **ceiling** operation rounds upward; this 3σ support rule is an implementation choice, not an assignment requirement. **Truncation** means omitting the Gaussian's infinite tails.

`astype(np.float64)` converts the stored intensities to 64-bit floating point without rescaling 0–255. It avoids rounding each filtering result back to `uint8`, which would obscure the later comparison. Set equal `sigmaX` and `sigmaY` for equal smoothing in both axes, and explicitly use reflected borders.

In [ ]:
# Apply one Gaussian pass to the original grayscale image using explicit support and precision.
gray_float = gray.astype(np.float64)
sigma_single = 2.0 * np.sqrt(2.0)
support_sigmas = 3.0
radius_single = int(np.ceil(support_sigmas * sigma_single))
kernel_single = 2 * radius_single + 1
gaussian_border = cv2.BORDER_REFLECT_101
gaussian_single = cv2.GaussianBlur(
    gray_float, (kernel_single, kernel_single),
    sigmaX=sigma_single, sigmaY=sigma_single, borderType=gaussian_border
)
print("Input source: original Task 1 grayscale image")
print(f"Standard deviation: 2*sqrt(2) = {sigma_single:.8f} pixels")
print(f"Variance: {sigma_single**2:.4f} pixels squared")
print(f"Support rule: radius = ceil({support_sigmas:g} * sigma)")
print(f"Kernel radius: {radius_single} pixels; kernel size: {kernel_single} x {kernel_single}")
print("Border handling: BORDER_REFLECT_101")
print(f"Input/output data type: {gray_float.dtype} / {gaussian_single.dtype}")
print(f"Output shape: {gaussian_single.shape}")
print(f"Observed output range: [{gaussian_single.min():.4f}, {gaussian_single.max():.4f}]")

**Interpretation:** This produces one smoothed floating-point image with unchanged spatial dimensions. Fractional values are retained for the subsequent two-pass comparison; the original `gray` array remains unchanged.

**Reference — documentation:** [OpenCV: Image Filtering](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html), `GaussianBlur` and `getGaussianKernel` — kernel size, standard deviations, normalized coefficients, and border handling.

### 2. Apply a Gaussian filter twice, using σ = 2 for each operation.

A **pass** is one application of a filter. Start from `gray_float`, the original Task 1 grayscale image converted to floating point, then filter the first result again. Filtering the original independently twice would not accumulate smoothing.

For ideal continuous Gaussian convolution, **variances add**:

$$\sigma_{\mathrm{effective}}^2=\sigma_1^2+\sigma_2^2=2^2+2^2=8,\qquad \sigma_{\mathrm{effective}}=2\sqrt{2}.$$

The standard deviations themselves do not add. This predicts agreement with Step 1 in the ideal continuous setting. Finite sampled kernels, borders, and numerical arithmetic can produce practical differences, investigated in Step 6.

Apply the same support rule $r=\lceil3\sigma\rceil$, giving a 13 × 13 kernel for each pass. Keep `float64` throughout and the same border rule as Step 1. The composed finite kernel has a wider support than either individual kernel; it is not simply a single 13 × 13 filter.

In [ ]:
# Apply two successive sigma-2 Gaussian passes to the original floating-point grayscale image.
sigma_pass = 2.0
radius_pass = int(np.ceil(support_sigmas * sigma_pass))
kernel_pass = 2 * radius_pass + 1
gaussian_pass1 = cv2.GaussianBlur(
    gray_float, (kernel_pass, kernel_pass),
    sigmaX=sigma_pass, sigmaY=sigma_pass, borderType=gaussian_border
)
gaussian_twice = cv2.GaussianBlur(
    gaussian_pass1, (kernel_pass, kernel_pass),
    sigmaX=sigma_pass, sigmaY=sigma_pass, borderType=gaussian_border
)
sigma_effective = np.sqrt(sigma_pass**2 + sigma_pass**2)
print("Pass 1 input: original Task 1 grayscale image converted to float64")
print("Pass 2 input: output of pass 1")
print(f"Sigma per pass: {sigma_pass:.1f} pixels; variance per pass: {sigma_pass**2:.1f} pixels squared")
print(f"Kernel per pass: {kernel_pass} x {kernel_pass}; radius: {radius_pass} pixels")
print(f"Ideal combined variance: {2 * sigma_pass**2:.1f} pixels squared")
print(f"Ideal effective sigma: {sigma_effective:.8f} pixels")
print("Border handling for both passes: BORDER_REFLECT_101")
for label, result in (("Pass 1", gaussian_pass1), ("Pass 2", gaussian_twice)):
    print(f"{label}: shape={result.shape}, dtype={result.dtype}, range=[{result.min():.4f}, {result.max():.4f}]")

**Interpretation:** `gaussian_twice` stores the sequential two-pass result. Its ideal effective standard deviation matches Step 1; numerical equality of the actual images remains to be tested in the following steps.

**Reference — documentation:** [OpenCV: Image Filtering, GaussianBlur](https://docs.opencv.org/4.x/d4/d86/group__imgproc__filter.html) — Gaussian filtering parameters. The variance-addition relation applies to ideal Gaussian convolution, not an exact guarantee for these finite implementations.

### 3. Display both resulting images.

Use the same grayscale colormap and **display limits** (`vmin=0`, `vmax=255`) for both floating-point results. The conversion to `float64` did not rescale the intensities to 0–1. Fixed limits avoid independent contrast stretching that could distort the visual comparison. Displaying the arrays does not round or overwrite their values.

In [ ]:
# Display the single-pass and two-pass Gaussian results using identical intensity scales.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)
gaussian_display_results = [
    (gaussian_single, f"One pass: sigma = 2*sqrt(2) ≈ {sigma_single:.4f}"),
    (gaussian_twice, "Two successive passes: sigma = 2 each")
]
for ax, (result, title) in zip(axes, gaussian_display_results):
    ax.imshow(result, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("Column (pixels)")
    ax.set_ylabel("Row (pixels)")
fig.suptitle("Gaussian filtering comparison — common intensity range: 0–255")
plt.tight_layout()
plt.show()

**Observation:** At this display scale, the two results appear very similar: both soften fine texture and edges while retaining the overall scene structure. Visual similarity does not establish pixel-wise equality; the next step computes their absolute difference.

**Reference — documentation:** [Matplotlib: imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) — scalar image display, colormaps, and intensity limits.